In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import warnings
warnings.filterwarnings('ignore')

ImportError: cannot import name 'DecisionTreeRegressor' from 'sklearn.ensemble' (c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\ensemble\__init__.py)

In [ ]:
# Load Dataset
# Dataset: Bike Sharing Dataset from UCI Repository
# Contains hourly bike rental data with weather and temporal features
df = pd.read_csv('hour.csv')
print(f"Dataset Shape: {df.shape}")
print(f"\nColumn Names:\n{df.columns.tolist()}")
df.head()

In [ ]:
# Rename Columns to Match Expected Format
column_mapping = {
    'dteday': 'datetime',
    'weathersit': 'weather',
    'hum': 'humidity',
    'mnth': 'month',
    'hr': 'hour',
    'yr': 'year',
    'cnt': 'count'
}
df = df.rename(columns=column_mapping)
print(f"Renamed columns: {df.columns.tolist()}")

In [ ]:

# Feature Engineering
# Parse datetime
df['datetime'] = pd.to_datetime(df['datetime'])

# Extract additional time features
df['day'] = df['datetime'].dt.day
df['dayofweek'] = df['datetime'].dt.dayofweek

# Decode year (UCI uses 0=2011, 1=2012)
df['year'] = df['year'].map({0: 2011, 1: 2012})

# Drop unnecessary columns
columns_to_drop = ['instant', 'datetime', 'casual', 'registered', 'atemp']
df = df.drop(columns_to_drop, axis=1)

print(f"Columns dropped: {columns_to_drop}")
print(f"Final columns: {df.columns.tolist()}")
print(f"Final shape: {df.shape}")


In [ ]:
# Check for Missing Values
print("Missing Values:")
print(df.isnull().sum())
print(f"\nTotal Missing: {df.isnull().sum().sum()}")

In [ ]:
# Data Information and Statistics
print("="*60)
print("DATA INFORMATION")
print("="*60)
df.info()
print("\n" + "="*60)
print("DESCRIPTIVE STATISTICS")
print("="*60)
df.describe()

In [ ]:

# Cell 5: Visualization 1 - Correlation Heatmap
plt.figure(figsize=(14, 12))
correlation_matrix = df.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.3f', 
            center=0, square=True, linewidths=0.5, 
            annot_kws={'size': 10})
plt.title('Correlation Heatmap - Bike Sharing Features', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Interpretation
print("""
FEATURE ENGINEERING INTERPRETATION FROM HEATMAP:
================================================

      
STRONGEST PREDICTORS OF 'count':
1. hour: ~0.40 - Peak demand during commute hours (8AM, 5-6PM)
2. temp: ~0.39 - Warmer weather increases demand
3. year: ~0.25 - Growth in popularity from 2011 to 2012

NEGATIVE CORRELATIONS:
1. humidity: ~-0.32 - High humidity reduces demand
2. weather: ~-0.13 - Worse weather conditions reduce demand

MULTICOLLINEARITY HANDLED:
- temp and atemp had 0.99 correlation -> Dropped atemp
- season and month have 0.83 correlation -> Both kept for now
  (season captures broader patterns, month captures specific effects)
""")

In [ ]:

# Visualization 2 - Target Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Histogram
sns.histplot(df['count'], bins=50, kde=True, color='steelblue', ax=axes[0])
axes[0].set_title('Distribution of Bike Rental Count', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Count', fontsize=10)
axes[0].set_ylabel('Frequency', fontsize=10)
axes[0].axvline(df['count'].mean(), color='red', linestyle='--', label=f'Mean: {df["count"].mean():.0f}')
axes[0].legend()

# Box plot
sns.boxplot(y=df['count'], color='steelblue', ax=axes[1])
axes[1].set_title('Box Plot of Bike Rental Count', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Count', fontsize=10)

plt.tight_layout()
plt.savefig('target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Target Statistics:")
print(f"  Mean: {df['count'].mean():.2f}")
print(f"  Median: {df['count'].median():.2f}")
print(f"  Std Dev: {df['count'].std():.2f}")
print(f"  Min: {df['count'].min()}")
print(f"  Max: {df['count'].max()}")
print(f"  Skewness: {df['count'].skew():.3f} (Right-skewed)")

In [ ]:
# Visualization 3 - Hourly Demand Pattern
# Critical for understanding temporal patterns
hourly_avg = df.groupby('hour')['count'].mean()
plt.figure(figsize=(12, 6))
plt.bar(hourly_avg.index, hourly_avg.values, color='steelblue', edgecolor='navy')
plt.xlabel('Hour of Day', fontsize=12)
plt.ylabel('Average Bike Rentals', fontsize=12)
plt.title('Average Bike Rentals by Hour of Day', fontsize=14, fontweight='bold')
plt.xticks(range(24))
plt.axvline(x=8, color='red', linestyle='--', label='Morning Commute')
plt.axvline(x=17, color='orange', linestyle='--', label='Evening Commute')
plt.legend()
plt.tight_layout()
plt.savefig('hourly_pattern.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Separate Features and Target
X = df.drop('count', axis=1)
y = df['count']
print(f"Feature columns: {X.columns.tolist()}")
print(f"Number of features: {len(X.columns)}")
print(f"Target variable: count")

In [ ]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

In [ ]:
# Standardize the Data
# Standardization is crucial for gradient descent optimization
# Features have different scales (hour 0-23, temp -10 to 50, humidity 0-100)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


print("Data Standardization Complete")
print(f"Training data mean (should be ~0): {X_train_scaled.mean(axis=0)[:5]}")
print(f"Training data std (should be ~1): {X_train_scaled.std(axis=0)[:5]}")

# Save scaler for API use
joblib.dump(scaler, 'scaler.pkl')
print("\nScaler saved to 'scaler.pkl'")

In [ ]:
# Scatter Plot BEFORE Training
# Shows raw data without prediction line
plt.figure(figsize=(12, 6))
temp_idx = list(X.columns).index('temp')
plt.scatter(X_test['temp'], y_test, alpha=0.4, color='steelblue', label='Actual Data Points')
plt.title('Bike Rentals vs Temperature (BEFORE Training)', fontsize=14, fontweight='bold')
plt.xlabel('Temperature (°C)', fontsize=12)
plt.ylabel('Bike Rental Count', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('before_training.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:

# Model 1 - Stochastic Gradient Descent (SGD) Linear Regression
# SGD is an optimization algorithm that updates weights using one sample at a time
# Learning rate and iterations are key hyperparameters
print("="*60)
print("MODEL 1: SGD LINEAR REGRESSION")
print("="*60)

sgd_model = SGDRegressor(
    max_iter=1,  # We'll iterate manually for loss tracking
    learning_rate='optimal',
    eta0=0.01,
    random_state=42,
    warm_start=True
)

train_losses_sgd = []
test_losses_sgd = []
epochs = 100

for epoch in range(epochs):
    sgd_model.fit(X_train_scaled, y_train)
    train_pred = sgd_model.predict(X_train_scaled)
    test_pred = sgd_model.predict(X_test_scaled)
    train_loss = mean_squared_error(y_train, train_pred)
    test_loss = mean_squared_error(y_test, test_pred)
    train_losses_sgd.append(train_loss)
    test_losses_sgd.append(test_loss)

print(f"Final Train MSE: {train_losses_sgd[-1]:.2f}")
print(f"Final Test MSE: {test_losses_sgd[-1]:.2f}")
print(f"R² Score: {r2_score(y_test, test_pred):.4f}")

In [ ]:
# Plot Loss Curve for SGD Linear Regression
plt.figure(figsize=(12, 6))
plt.plot(range(1, epochs+1), train_losses_sgd, 'b-', label='Train Loss (MSE)', linewidth=2)
plt.plot(range(1, epochs+1), test_losses_sgd, 'r-', label='Test Loss (MSE)', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Mean Squared Error (MSE)', fontsize=12)
plt.title('Loss Curve - SGD Linear Regression', fontsize=14, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('sgd_loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Model 2 - Random Forest Regressor
# Ensemble method that combines multiple decision trees
# Reduces overfitting through averaging
print("="*60)
print("MODEL 2: RANDOM FOREST REGRESSOR")
print("="*60)

rf_model = RandomForestRegressor(
    n_estimators=100,  # Number of trees
    max_depth=10,       # Prevent overfitting
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_scaled, y_train)

rf_train_pred = rf_model.predict(X_train_scaled)
rf_test_pred = rf_model.predict(X_test_scaled)
rf_train_loss = mean_squared_error(y_train, rf_train_pred)
rf_test_loss = mean_squared_error(y_test, rf_test_pred)

print(f"Train MSE: {rf_train_loss:.2f}")
print(f"Test MSE: {rf_test_loss:.2f}")
print(f"R² Score: {r2_score(y_test, rf_test_pred):.4f}")

In [ ]:
# Cell 18: Model 3 - Decision Tree Regressor
# Single tree that can capture non-linear relationships
# Prone to overfitting without proper constraints
print("="*60)
print("MODEL 3: DECISION TREE REGRESSOR")
print("="*60)

dt_model = DecisionTreeRegressor(
    max_depth=8,          # Limit depth to prevent overfitting
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)
dt_model.fit(X_train_scaled, y_train)

dt_train_pred = dt_model.predict(X_train_scaled)
dt_test_pred = dt_model.predict(X_test_scaled)
dt_train_loss = mean_squared_error(y_train, dt_train_pred)
dt_test_loss = mean_squared_error(y_test, dt_test_pred)


print(f"Train MSE: {dt_train_loss:.2f}")
print(f"Test MSE: {dt_test_loss:.2f}")
print(f"R² Score: {r2_score(y_test, dt_test_pred):.4f}")

In [ ]:
# Model Comparison
print("="*60)
print("MODEL COMPARISON SUMMARY")
print("="*60)

models = {
    'SGD Linear Regression': {
        'train_loss': train_losses_sgd[-1],
        'test_loss': test_losses_sgd[-1],
        'model': sgd_model
    },
    'Random Forest': {
        'train_loss': rf_train_loss,
        'test_loss': rf_test_loss,
        'model': rf_model
    },
    'Decision Tree': {
        'train_loss': dt_train_loss,
        'test_loss': dt_test_loss,
        'model': dt_model
    }
}

print(f"\n{'Model':<30} {'Train MSE':>15} {'Test MSE':>15}")
print("-" * 60)
for name, data in models.items():
    print(f"{name:<30} {data['train_loss']:>15.2f} {data['test_loss']:>15.2f}")

In [ ]:
# Select and Save Best Model
best_model_name = min(models, key=lambda x: models[x]['test_loss'])
best_model = models[best_model_name]['model']
best_test_loss = models[best_model_name]['test_loss']

print(f"\n{'='*60}")
print(f"BEST MODEL: {best_model_name}")
print(f"Test MSE: {best_test_loss:.2f}")
print(f"{'='*60}")

# Save best model
joblib.dump(best_model, 'best_model.pkl')
print("\nBest model saved as 'best_model.pkl'")

# Save feature names for API consistency
feature_names = list(X.columns)
joblib.dump(feature_names, 'feature_names.pkl')
print(f"Feature names saved: {feature_names}")

In [ ]:
# Scatter Plot AFTER Training with Linear Regression Line
plt.figure(figsize=(12, 6))

# Scatter actual test data
plt.scatter(X_test['temp'], y_test, alpha=0.4, color='steelblue', label='Actual Data Points')

# Plot SGD Linear Regression fitted line
sgd_test_pred = sgd_model.predict(X_test_scaled)
sort_idx = np.argsort(X_test['temp'].values)
plt.plot(X_test['temp'].values[sort_idx], sgd_test_pred[sort_idx], 
         color='red', linewidth=3, label='SGD Linear Regression Line')

plt.title('Bike Rentals vs Temperature (AFTER Training)', fontsize=14, fontweight='bold')
plt.xlabel('Temperature (°C)', fontsize=12)
plt.ylabel('Bike Rental Count', fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('after_training.png', dpi=150, bbox_inches='tight')
plt.show() 

In [ ]:
# Feature Importance (from Random Forest)
feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Feature Importance (Random Forest):")
print(feature_importance.to_string(index=False))

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance, palette='viridis')
plt.title('Feature Importance - Random Forest', fontsize=14, fontweight='bold')
plt.xlabel('Importance', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Prediction Function for API Use
def make_prediction(features_dict):
    """
    Make a prediction using the best model.
    
    Parameters:
    -----------
    features_dict : dict
        Dictionary containing all required features with keys:
        'season', 'holiday', 'workingday', 'weather', 'temp', 
        'humidity', 'windspeed', 'hour', 'day', 'month', 'year', 'dayofweek'
    
    Returns:
    --------
    float : Predicted bike rental count
    """
    # Create DataFrame with correct column order
    input_data = pd.DataFrame([features_dict], columns=feature_names)

    # Scale the input using saved scaler
    input_scaled = scaler.transform(input_data)
    
    # Make prediction
    prediction = best_model.predict(input_scaled)
    
    # Ensure non-negative (can't have negative rentals)
    return max(0, float(prediction[0])) 

In [ ]:
# Test Prediction on Single Row from Test Data
test_row = X_test.iloc[0]
test_row_dict = test_row.to_dict()
actual_value = y_test.iloc[0]
predicted_value = make_prediction(test_row_dict)

print("="*60)
print("SINGLE PREDICTION TEST")
print("="*60)
print(f"\nInput Features:")
for key, value in test_row_dict.items():
    print(f"  {key}: {value}")
print(f"\nActual Value: {actual_value}")
print(f"Predicted Value: {predicted_value:.2f}")
print(f"Absolute Error: {abs(actual_value - predicted_value):.2f}")
print(f"Percentage Error: {abs(actual_value - predicted_value)/actual_value * 100:.2f}%")

In [ ]:
# Cell 25: Test with Custom Input (Simulating API Call)
print("="*60)
print("CUSTOM PREDICTION TEST (API Simulation)")
print("="*60)

custom_input = {
    'season': 2,        # Summer
    'holiday': 0,       # Not a holiday
    'workingday': 1,    # Working day
    'weather': 1,       # Clear weather
    'temp': 25.0,       # 25°C
    'humidity': 50.0,   # 50% humidity
    'windspeed': 10.0,  # Low wind
    'hour': 17,         # 5 PM (evening commute)
    'day': 15,          # Mid-month
    'month': 7,         # July
    'year': 2024,       # Current year
    'dayofweek': 1      # Tuesday
}


predicted = make_prediction(custom_input)
print(f"\nInput: Clear summer evening commute at 25°C")
print(f"Predicted Bike Rentals: {predicted:.0f}")

In [ ]:
# Cell 26: Summary Statistics for Video Demo
print("="*60)
print("FINAL SUMMARY FOR VIDEO DEMO")
print("="*60)
print(f"""
DATASET: Bike Sharing Demand (Kaggle)
- Records: {len(df)}
- Features: {len(feature_names)}
- Target: count (bike rentals)

MODEL PERFORMANCE (Test MSE):
- SGD Linear Regression: {models['SGD Linear Regression']['test_loss']:.2f}
- Random Forest: {models['Random Forest']['test_loss']:.2f}
- Decision Tree: {models['Decision Tree']['test_loss']:.2f}

BEST MODEL: {best_model_name}
- Test MSE: {best_test_loss:.2f}

LOSS INTERPRETATION:
- MSE of {best_test_loss:.2f} means average squared error is {np.sqrt(best_test_loss):.2f} bikes
- This is relatively LOW given the range of bike rentals (0-977)

FEATURE ENGINEERING DECISIONS:
1. Dropped 'atemp' (0.98 correlation with 'temp')
2. Dropped 'casual' and 'registered' (data leakage)
3. Extracted hour, day, month, year, dayofweek from datetime
4. Standardized all features for gradient descent
""")  